# 09 - Silver 4: outage intervals, impacted regions, affected customers
The pack has no outage table and no customer table, so the domain rule needs documented assumptions.
**Confirm all three with your mentor and write them in the decision log.**
1. **Outage reading** = a reading whose CURRENT status (after CDC) is FAILED.
2. **Outage interval** (per tower) = a run of consecutive FAILED readings. It starts at the first FAILED reading and ends at
   the next reading of that tower (recovery seen). No later reading -> open interval, minutes left null.
3. **Affected customers** = distinct operator_id on the FAILED readings (closest customer-side key in the data).
4. **Impacted regions** come from the region_id on each FAILED reading. In this data a tower appears under many regions,
   so intervals are per tower and regions are listed on them.

### Settings (names of your catalog and schema)

In [0]:
from pyspark.sql import functions as F, Window as W

cat = "capstone6_telecom.telecom"

EXTREME_LATENCY_MS = 500                      # TC02: exactly 35 rows are above this
EVENT_STATUS = ["OPEN", "FAILED", "PENDING", "COMPLETED"]
CDC_STATUS   = EVENT_STATUS + ["CANCELLED", "REVERSED"]
SEC_TYPES    = ["CHECK", "ALERT", "REVIEW"]
OUTAGE_STATUS = "FAILED"                      # a reading counts as an outage when its CURRENT status is this

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.silver_outage_intervals AS
WITH r AS (
  SELECT tower_id, region_id, measurement_id, operator_id, event_ts, has_alert,
         (current_status = '{OUTAGE_STATUS}') AS is_out,
         LAG(current_status = '{OUTAGE_STATUS}') OVER w AS prev_out,
         LEAD(event_ts) OVER w AS next_ts
  FROM {cat}.silver_network_enriched
  WINDOW w AS (PARTITION BY tower_id ORDER BY event_ts, measurement_id)
), g AS (
  SELECT *, SUM(CASE WHEN is_out AND NOT COALESCE(prev_out, false) THEN 1 ELSE 0 END)
            OVER (PARTITION BY tower_id ORDER BY event_ts, measurement_id) AS grp
  FROM r
)
SELECT concat(tower_id, '|', date_format(MIN(event_ts), 'yyyyMMddHHmmss')) AS outage_id,
       tower_id,
       MIN(event_ts)                              AS start_ts,
       MAX(event_ts)                              AS last_failed_ts,
       MAX_BY(next_ts, event_ts)                  AS end_ts,
       MAX_BY(next_ts, event_ts) IS NULL          AS is_open,
       CASE WHEN MAX_BY(next_ts, event_ts) IS NULL THEN NULL
            ELSE (unix_timestamp(MAX_BY(next_ts, event_ts)) - unix_timestamp(MIN(event_ts))) / 60.0 END AS outage_minutes,
       COUNT(*)                                   AS failed_readings,
       COLLECT_SET(region_id)                     AS impacted_regions,
       SIZE(COLLECT_SET(region_id))               AS impacted_region_count,
       COUNT(DISTINCT operator_id)                AS affected_customers,
       MAX(CASE WHEN has_alert THEN 1 ELSE 0 END) = 1 AS alert_corroborated
FROM g WHERE is_out
GROUP BY tower_id, grp
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {cat}.silver_outage_region_daily AS
SELECT region_id,
       CAST(event_ts AS DATE)           AS outage_date,
       COUNT(*)                         AS failed_readings,
       COUNT(DISTINCT tower_id)         AS impacted_towers,
       COUNT(DISTINCT operator_id)      AS affected_customers
FROM {cat}.silver_network_enriched
WHERE current_status = '{OUTAGE_STATUS}'
GROUP BY region_id, CAST(event_ts AS DATE)
""")

## Checks

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS intervals,                              -- about 2,108 when computed in pandas
       SUM(CAST(is_open AS INT)) AS open_intervals,        -- about 21
       SUM(failed_readings) AS failed_readings,            -- must equal current FAILED readings (2,755)
       ROUND(SUM(outage_minutes), 0) AS closed_outage_minutes
FROM {cat}.silver_outage_intervals"""))

display(spark.sql(f"""SELECT COUNT(*) AS current_failed FROM {cat}.silver_network_enriched WHERE current_status='{OUTAGE_STATUS}'"""))

# No tower may have two overlapping intervals
display(spark.sql(f"""
SELECT COUNT(*) AS overlapping_pairs FROM {cat}.silver_outage_intervals a
JOIN {cat}.silver_outage_intervals b ON a.tower_id = b.tower_id AND a.outage_id < b.outage_id
 AND a.start_ts < COALESCE(b.end_ts, TIMESTAMP'9999-12-31') AND b.start_ts < COALESCE(a.end_ts, TIMESTAMP'9999-12-31')"""))   # expect 0

display(spark.table(f"{cat}.silver_outage_intervals").orderBy("start_ts").limit(10))